# Logistic Regression 

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [16]:
class LogisticRegression:
    def __init__(self, lr=0.01, n_iter=5000, C=1.0, weight_fraud=1.0, threshold=0.3):
        self.lr = lr                    # learning rate
        self.n_iter = n_iter            # number of GD steps
        self.C = C                      # inverse regularisation strength (L2)
        self.w = None
        self.b = 0.0
        self.weight_fraud = weight_fraud
        self.threshold = threshold

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        for _ in tqdm(range(self.n_iter), desc='LogReg training', leave=False):
            linear = X @ self.w + self.b
            prob = self._sigmoid(linear)

            error = prob - y
            weights = np.where(y == 1, self.weight_fraud, 1.0)
            dw = (1 / n_samples) * (X.T @ (error * weights)) + (1 / self.C) * self.w
            db = (1 / n_samples) * np.sum(error * weights)
            
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X):
        linear = X @ self.w + self.b
        prob = self._sigmoid(linear)
        return np.column_stack([1 - prob, prob])

    def predict(self, X):
        prob = self._sigmoid(X @ self.w + self.b)
        return (prob >= self.threshold).astype(int)

In [ ]:
# Metric helpers 
def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))
def precision(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0
def recall(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0
def f1(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

In [ ]:
# Load test data 
test_df = pd.read_csv('data/data_encoded_test.csv')
X_test = test_df.drop(columns=['is_fraud']).values
y_test = test_df['is_fraud'].values

df = pd.read_csv('data/data_encoded_train.csv')
X_train = df.drop(columns=['is_fraud']).values
y_train = df['is_fraud'].values
lr = LogisticRegression(C=0.1,weight_fraud=50.0,threshold= 0.3)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print("Final Metrics:")
print(f'accuracy: {accuracy(y_test,y_pred)}')
print(f'recall: {recall(y_test,y_pred)}')
print(f'f1 score: {f1(y_test,y_pred)}')
print(f'precision: {precision(y_test,y_pred)}')

Final Metrics:
accuracy: 0.9038
recall: 0.7777777777777778
f1 score: 0.05500982318271121
precision: 0.028513238289205704
